<!-- m6:clase=4;checkpoints=7;preguntas=3 -->
# Módulo 6 · Clase 4 — Redes neuronales con TensorFlow y Keras · Calidad de software (QA) para ML
**Diplomado en Big Data e Inteligencia Artificial · CEC-EPN** · Ing. Eduardo Avilés · lunes 16-11-2026 · 18:30–21:30

**Cómo trabajar este cuaderno**
1. Ejecute las celdas en orden con `Shift + Enter`. Las celdas **➕ Ejemplo adicional** son opcionales.
2. En cada **✏️ Checkpoint** complete los `___` y ejecute: si está bien, aparece ✅ y un código personal.
3. Responda las **💬 Preguntas** en 2 o 3 líneas.
4. Al final: **Archivo → Guardar (Ctrl+S)** → en *Repositorio* elija **el suyo** → ruta `talleres/clase4.ipynb` → mensaje `Clase 4`.

> Datos: app de pagos móviles **PagoYa** (ficticia, simulada con fines didácticos).

## 0. Preparación y registro (Checkpoint 0)

In [ ]:
import importlib, subprocess, sys                            # instala lo que falte (solo la primera vez)
for modulo, paquete in [('pytest_cov', 'pytest-cov'), ('tensorflow', 'tensorflow')]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paquete])
print('Paquetes listos')

In [ ]:
import os, json, time, warnings, urllib.request
URL = 'https://raw.githubusercontent.com/JavyAviles/m6-ia-plantilla/main/curso.py'
if not os.path.exists('curso.py'):
    urllib.request.urlretrieve(URL, 'curso.py')
import curso
warnings.filterwarnings('ignore'); os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # menos mensajes de TensorFlow
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf                                     # cálculo con tensores y gradientes
from tensorflow import keras                                # API de alto nivel para redes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
plt.rcParams['figure.figsize'] = (10, 3.4)
print('Listo · curso', curso.VERSION, '· TensorFlow', tf.__version__)

### ✏️ Checkpoint 0 · ⏱ 2 min
Registre su nombre como **'Apellido Nombre'** e indique la **clase 4**.

In [ ]:
curso.registrar('___', clase=___)          # ✏️ su apellido y nombre, y el número de esta clase

In [ ]:
df = curso.cargar_pagoya_limpio()                           # mismos datos de siempre
df['ratio'] = df['monto'] / df['ingreso_mensual']
cols = curso.X_COLS + ['ratio']                             # 9 variables
X_train, X_test, y_train, y_test = train_test_split(df[cols], df['fraude'], test_size=0.3,
                                                    stratify=df['fraude'], random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
escalador = StandardScaler().fit(X_tr)                      # las redes NECESITAN datos escalados
X_tr_esc, X_val_esc, X_test_esc = escalador.transform(X_tr), escalador.transform(X_val), escalador.transform(X_test)
print('Entrenamiento:', X_tr_esc.shape, '· validación:', X_val_esc.shape, '· prueba:', X_test_esc.shape)

---
## 4.1 Redes neuronales: de una neurona al deep learning
Una **neurona** calcula una suma ponderada de sus entradas más un sesgo, z = w·x + b, y le aplica una **función de activación**. Con la sigmoide, una neurona sola **es** una regresión logística. Una red apila **capas** de neuronas: cada capa transforma la representación de la anterior.

### ✏️ Checkpoint 1 · ⏱ 4 min
Calcule la salida de una neurona con entradas `x = [0.5, -1.2, 2.0]`, pesos `w = [0.8, 0.3, -0.5]`, sesgo `b = 0.1` y activación **sigmoide**.

In [ ]:
x = np.array([0.5, -1.2, 2.0]); w = np.array([0.8, 0.3, -0.5]); b = 0.1
z = np.___(w, x) + ___                                         # ✏️ suma ponderada
salida = 1 / (1 + np.exp(___))                                # ✏️ sigmoide
print(f'z = {z:.3f} · salida = {salida:.4f}')
curso.checkpoint(1, salida)

### ➕ Ejemplo adicional: funciones de activación

In [ ]:
z = np.linspace(-5, 5, 200)
activaciones = {'sigmoide': 1 / (1 + np.exp(-z)), 'tanh': np.tanh(z), 'ReLU': np.maximum(0, z)}
for nombre, a in activaciones.items():
    plt.plot(z, a, label=nombre, lw=2)
plt.axhline(0, color='gray', lw=0.5); plt.legend(); plt.title('Funciones de activación'); plt.show()

**Retropropagación**: la red se entrena minimizando una **función de pérdida** (entropía cruzada en clasificación) con **descenso de gradiente**: cada peso se mueve en contra de su gradiente, w ← w − η · ∂pérdida/∂w. La retropropagación calcula esos gradientes de atrás hacia adelante con la **regla de la cadena**.

---
## 4.2 TensorFlow: tensores, gradiente automático y costo de una red
Un **tensor** es un arreglo de n dimensiones (escalar, vector, matriz…). TensorFlow sabe **derivar automáticamente** cualquier cálculo hecho con tensores: eso es lo que hace posible entrenar redes con millones de pesos.

In [ ]:
t = tf.constant([[1.0, 2.0], [3.0, 4.0]])                    # tensor 2 × 2
print('forma:', t.shape, '· tipo:', t.dtype)
print('producto matricial:\n', (t @ t).numpy())
print('dispositivo disponible:', 'GPU' if tf.config.list_physical_devices('GPU') else 'CPU')

### ✏️ Checkpoint 2 · ⏱ 5 min
Use `tf.GradientTape` para calcular **∂pérdida/∂w** de una neurona con x = 2, y = 1, w = 0,5, b = 0 y pérdida de entropía cruzada.

In [ ]:
w = tf.Variable(0.5); b = tf.Variable(0.0); x, y = 2.0, 1.0
with tf.GradientTape() as cinta:                             # 'graba' las operaciones
    p = tf.sigmoid(w * x + b)
    perdida = -(y * tf.math.log(p) + (1 - y) * tf.math.log(1 - p))
grad = cinta.___(perdida, ___)                            # ✏️ derivada respecto a w
print(f'p = {p.numpy():.4f} · gradiente = {grad.numpy():.4f} · fórmula (p − y)·x = {(p.numpy() - y) * x:.4f}')
curso.checkpoint(2, grad.numpy())

### ➕ Ejemplo adicional: costo computacional (CPU frente a GPU)

In [ ]:
for n in [500, 1000, 2000]:                                  # multiplicar matrices n × n: costo ~ n³
    a = tf.random.normal((n, n)); t0 = time.perf_counter(); _ = (a @ a).numpy()
    print(f'n = {n:>5} → {time.perf_counter() - t0:.4f} s')

### ✏️ Checkpoint 3 · ⏱ 3 min
¿Cuántos **parámetros** (pesos + sesgos) tiene una red con **9 entradas → 16 neuronas → 8 neuronas → 1 salida**?

In [ ]:
parametros = (9 * 16 + ___) + (16 * 8 + ___) + (8 * 1 + ___)       # ✏️ entradas × neuronas + sesgos, por capa
print('Parámetros:', parametros)
curso.checkpoint(3, parametros)

---
## 4.3 Keras: construir, entrenar y vigilar una red
**Keras** es la API de alto nivel de TensorFlow: se definen capas, se **compila** (optimizador, pérdida, métricas) y se **entrena** con `fit`. Los **callbacks** actúan durante el entrenamiento: `EarlyStopping` (detenerse cuando la validación deja de mejorar), `ModelCheckpoint` (guardar el mejor modelo) y `TensorBoard` (registrar métricas).

In [ ]:
keras.utils.set_random_seed(42)                             # reproducibilidad (semillas de Python, NumPy y TF)
red = keras.Sequential([
    keras.Input(shape=(len(cols),)),                         # 9 variables escaladas
    keras.layers.Dense(16, activation='relu'),               # capa oculta 1
    keras.layers.Dropout(0.2),                               # apaga 20 % de neuronas al entrenar: regulariza
    keras.layers.Dense(8, activation='relu'),                # capa oculta 2
    keras.layers.Dense(1, activation='sigmoid'),             # salida: probabilidad de fraude
])
red.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy',
            metrics=[keras.metrics.AUC(name='auc')])
red.summary()
print('Parámetros:', red.count_params())

In [ ]:
peso_fraude = (y_tr == 0).sum() / (y_tr == 1).sum()          # ~11: un fraude pesa como 11 normales
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('mejor_red.keras', monitor='val_auc', mode='max', save_best_only=True),
]
if importlib.util.find_spec('tensorboard') is not None:     # registra métricas para TensorBoard (viene en Colab)
    callbacks.append(keras.callbacks.TensorBoard(log_dir='logs'))
t0 = time.perf_counter()
hist = red.fit(X_tr_esc, y_tr, validation_data=(X_val_esc, y_val), epochs=200, batch_size=64,
               class_weight={0: 1.0, 1: peso_fraude}, callbacks=callbacks, verbose=0)
print(f'Épocas entrenadas: {len(hist.history["loss"])} · tiempo {time.perf_counter() - t0:.1f} s')
plt.plot(hist.history['auc'], label='entrenamiento'); plt.plot(hist.history['val_auc'], label='validación')
plt.xlabel('época'); plt.ylabel('AUC'); plt.legend(); plt.title('Curvas de aprendizaje'); plt.show()

### ✏️ Checkpoint 4 · ⏱ 4 min
Calcule el **AUC de la red en prueba** con `roc_auc_score` y compárelo con una regresión logística.

In [ ]:
p_red = red.predict(___, verbose=0).ravel()          # ✏️ probabilidades de la red
auc_red = roc_auc_score(y_test, p_red)
log = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_tr_esc, y_tr)
auc_log = roc_auc_score(y_test, log.predict_proba(X_test_esc)[:, 1])
print(f'AUC red: {auc_red:.3f} · AUC logística: {auc_log:.3f}')
curso.checkpoint(4, auc_red)

### ➕ Ejemplo adicional: API funcional (dos ramas de entrada)

In [ ]:
entrada_num = keras.Input(shape=(len(cols) - 1,), name='numericas')   # 8 variables continuas
entrada_disp = keras.Input(shape=(1,), name='dispositivo')            # 1 variable binaria
rama = keras.layers.Dense(16, activation='relu')(entrada_num)
unidas = keras.layers.Concatenate()([rama, entrada_disp])             # se unen las ramas
salida = keras.layers.Dense(1, activation='sigmoid')(keras.layers.Dense(8, activation='relu')(unidas))
red_func = keras.Model(inputs=[entrada_num, entrada_disp], outputs=salida)
red_func.summary()

### ➕ Ejemplo adicional: TensorBoard (solo en Colab)

In [ ]:
if 'google.colab' in sys.modules:                           # panel interactivo de métricas por época
    get_ipython().run_line_magic('load_ext', 'tensorboard')
    get_ipython().run_line_magic('tensorboard', '--logdir logs')
else:
    print('TensorBoard se abre en Colab con: %load_ext tensorboard  y  %tensorboard --logdir logs')

In [ ]:
import joblib                                               # guardar y servir el modelo
red.save('red_fraude.keras'); joblib.dump(escalador, 'escalador.joblib')   # los DOS artefactos
red_cargada = keras.models.load_model('red_fraude.keras'); esc_cargado = joblib.load('escalador.joblib')
def predecir(transaccion):
    """De una transacción (dict) a la probabilidad de fraude: lo que llamaría la app."""
    x = pd.DataFrame([transaccion])[cols]
    return float(red_cargada.predict(esc_cargado.transform(x), verbose=0)[0, 0])
ejemplo = X_test.iloc[0].to_dict()
print('Transacción:', {k: round(v, 2) for k, v in ejemplo.items()})
print(f'Probabilidad de fraude: {predecir(ejemplo):.3f}')

### 💬 Pregunta 1
En PagoYa la red supera a la logística en AUC. Frente a un ensamble de árboles como XGBoost (Clase 2), ¿cuándo vale la pena una red neuronal y cuándo no?

**Respuesta:** ___

---
## 4.4 Calidad de software (QA): la pirámide de pruebas
| Nivel | Qué prueba | Ejemplo en PagoYa | Cantidad y velocidad |
|---|---|---|---|
| **Unitarias** | Una función aislada | `semaforo(0.8) == 'BLOQUEADA'` | Muchas · milisegundos |
| **Integración** | Que varias piezas funcionen juntas | API → JSON → DataFrame limpio | Algunas · segundos |
| **De extremo a extremo (E2E)** | El flujo completo como lo vive el usuario | transacción → modelo → semáforo en la app | Pocas · lentas y frágiles |

Además: **regresión** (lo que funcionaba sigue funcionando), **humo** (*smoke*: lo mínimo arranca), **rendimiento** (tiempo de respuesta) y **aceptación** (cumple lo que pidió el negocio). **TDD**: escribir la prueba **antes** del código. **Cobertura**: % de líneas del código que las pruebas ejecutan.

In [ ]:
%%writefile pagoya_utils.py
"""Funciones de negocio de la app PagoYa (módulo que vamos a probar)."""
import numpy as np


def calcular_ratio(monto, ingreso):
    """Monto relativo al ingreso del usuario."""
    if ingreso <= 0:
        raise ValueError("El ingreso debe ser positivo")
    if monto < 0:
        raise ValueError("El monto no puede ser negativo")
    return monto / ingreso


def semaforo(p, umbral_revisar=0.2, umbral_bloquear=0.5):
    """Decisión de la app según la probabilidad de fraude."""
    if not 0 <= p <= 1:
        raise ValueError("p debe estar entre 0 y 1")
    if p < umbral_revisar:
        return 'APROBADA'
    if p < umbral_bloquear:
        return 'REVISAR'
    return 'BLOQUEADA'


def validar_transaccion(t):
    """Devuelve la lista de errores de una transacción (vacía si es válida)."""
    errores = []
    for campo in ['monto', 'ingreso_mensual', 'distancia_km', 'dispositivo_nuevo', 'intentos_pin']:
        if campo not in t:
            errores.append(f'falta {campo}')
    if t.get('monto', 0) <= 0:
        errores.append('monto no positivo')
    if t.get('intentos_pin', 1) not in (1, 2, 3):
        errores.append('intentos_pin fuera de rango')
    if t.get('dispositivo_nuevo', 0) not in (0, 1):
        errores.append('dispositivo_nuevo debe ser 0 o 1')
    return errores


def costo_decision(y_real, bloquear, montos, costo_bloqueo=5):
    """Costo total: fraudes no bloqueados (su monto) + bloqueos de transacciones legítimas."""
    y_real, bloquear, montos = np.asarray(y_real), np.asarray(bloquear, bool), np.asarray(montos, float)
    if not (len(y_real) == len(bloquear) == len(montos)):
        raise ValueError("Las tres listas deben tener el mismo largo")
    fraudes_no_detectados = np.sum(montos * (y_real == 1) * (~bloquear))
    falsas_alarmas = costo_bloqueo * np.sum((y_real == 0) & bloquear)
    return float(fraudes_no_detectados + falsas_alarmas)


def formatear_usd(x):
    """Texto en dólares con separador de miles."""
    return f"USD {x:,.2f}"


def psi(esperado, actual, cortes=10):
    """Population Stability Index entre dos muestras de una variable."""
    esperado, actual = np.asarray(esperado, float), np.asarray(actual, float)
    bordes = np.unique(np.quantile(esperado, np.linspace(0, 1, cortes + 1)))
    bordes[0], bordes[-1] = -np.inf, np.inf
    pe = np.histogram(esperado, bordes)[0] / len(esperado)
    pa = np.histogram(actual, bordes)[0] / len(actual)
    pe, pa = np.clip(pe, 1e-6, None), np.clip(pa, 1e-6, None)
    return float(np.sum((pa - pe) * np.log(pa / pe)))


In [ ]:
%%writefile test_pagoya_utils.py
"""Pruebas unitarias de pagoya_utils con pytest."""
import pytest
from pagoya_utils import calcular_ratio, semaforo, validar_transaccion


def test_ratio_basico():                                     # caso normal
    assert calcular_ratio(100, 400) == 0.25


def test_ratio_ingreso_cero_lanza_error():                   # caso de error: debe lanzar excepción
    with pytest.raises(ValueError):
        calcular_ratio(100, 0)


@pytest.mark.parametrize("p, esperado", [(0.05, 'APROBADA'), (0.35, 'REVISAR'), (0.80, 'BLOQUEADA')])
def test_semaforo(p, esperado):                              # una prueba, tres casos
    assert semaforo(p) == esperado


def test_transaccion_valida_sin_errores():
    t = {'monto': 50, 'ingreso_mensual': 900, 'distancia_km': 3, 'dispositivo_nuevo': 0, 'intentos_pin': 1}
    assert validar_transaccion(t) == []

In [ ]:
!python -m pytest -q test_pagoya_utils.py --cov=pagoya_utils --cov-report=term-missing

In [ ]:
%%writefile test_validacion.py
from pagoya_utils import validar_transaccion


def test_transaccion_invalida_reporta_errores():             # cubre las ramas de error
    errores = validar_transaccion({'monto': -10, 'intentos_pin': 5, 'dispositivo_nuevo': 2})
    assert 'monto no positivo' in errores
    assert 'intentos_pin fuera de rango' in errores

In [ ]:
!python -m pytest -q test_pagoya_utils.py test_validacion.py --cov=pagoya_utils --cov-report=term-missing

### ✏️ Checkpoint 5 · ⏱ 6 min
Complete la prueba de `costo_decision`: un fraude de USD 200 que **no** se bloqueó y una transacción legítima de USD 30 que **sí** se bloqueó (costo de bloqueo USD 5). ¿Cuánto debe costar? Ejecute y alcance **cobertura ≥ 70 %**.

In [ ]:
%%writefile test_costo.py
from pagoya_utils import costo_decision


def test_costo_fraude_no_detectado_y_falsa_alarma():
    costo = costo_decision(y_real=[1, 0, 0], bloquear=[False, True, False], montos=[200, 30, 40])
    assert costo == ___                                          # ✏️ fraude no detectado + falsa alarma

In [ ]:
!python -m pytest -q test_pagoya_utils.py test_validacion.py test_costo.py --cov=pagoya_utils --cov-report=term-missing --cov-report=json:cobertura.json
cobertura = json.load(open('cobertura.json'))['totals']['percent_covered']   # lee el % del reporte
print(f'Cobertura total: {cobertura:.1f} %')
curso.checkpoint(5, cobertura)

### ➕ Ejemplo adicional: pruebas de integración y de extremo a extremo (E2E)

In [ ]:
%%writefile test_integracion.py
import curso
from pagoya_utils import semaforo


def test_api_a_dataframe():                                  # INTEGRACIÓN: API + JSON + limpieza
    df = curso.cargar_pagoya_limpio()
    assert len(df) > 2900
    assert df['evento_id'].is_unique                         # sin duplicados
    assert (df['monto'] > 0).all()                           # sin montos negativos
    assert df['edad'].notna().all()                          # sin nulos


def test_flujo_completo_pantalla():                          # E2E simplificado: probabilidad → decisión de la app
    for p in [0.0, 0.19, 0.2, 0.49, 0.5, 1.0]:
        assert semaforo(p) in {'APROBADA', 'REVISAR', 'BLOQUEADA'}

In [ ]:
!python -m pytest -q test_integracion.py

### 💬 Pregunta 2
¿Por qué la pirámide tiene muchas pruebas unitarias y pocas E2E? ¿Qué pasaría con un equipo que solo tiene pruebas E2E?

**Respuesta:** ___

---
## 4.5 QA para ML y MLOps: datos, modelos, reproducibilidad y deriva
En ML hay que probar **tres cosas**: el **código** (lo anterior), los **datos** y el **modelo**.

In [ ]:
def pruebas_de_datos(d):
    """Pruebas de DATOS: esquema, nulos, rangos y valores permitidos."""
    return {'columnas esperadas': set(cols + ['fraude']).issubset(d.columns),
            'sin nulos': bool(d[cols].notna().all().all()),
            'montos positivos': bool((d['monto'] > 0).all()),
            'fraude es 0/1': set(d['fraude'].unique()) <= {0, 1},
            'intentos_pin en 1..3': bool(d['intentos_pin'].between(1, 3).all())}
pd.Series(pruebas_de_datos(df), name='¿pasa?')

In [ ]:
base = X_test.iloc[[0]].copy(); base['dispositivo_nuevo'] = 1
lejos = base.copy(); lejos['distancia_km'] = 60              # misma transacción, mucho más lejos
otra_edad = base.copy(); otra_edad['edad'] = base['edad'] + 1   # misma transacción, un año más
p = lambda x: float(red.predict(escalador.transform(x[cols]), verbose=0)[0, 0])
pruebas_modelo = {
    'rendimiento: AUC red ≥ AUC logística − 0,03': auc_red >= auc_log - 0.03,
    'direccional: más distancia → más riesgo': p(lejos) > p(base),
    'invariancia: +1 año de edad cambia < 5 pts': abs(p(otra_edad) - p(base)) < 0.05,
}
print({'riesgo base': round(p(base), 3), 'a 60 km': round(p(lejos), 3), '+1 año': round(p(otra_edad), 3)})
pd.Series(pruebas_modelo, name='¿pasa?')

**Reproducibilidad: un caso real de este curso.** En la Clase 2, el mismo código de random forest dio 0,904 en una máquina y **0,9557 en Colab**: la versión de scikit-learn era distinta. Por eso se **fijan las versiones** (`requirements.txt`) y las semillas.

In [ ]:
!pip freeze | grep -iE "^(tensorflow|scikit-learn|numpy|pandas|xgboost)==" | tee requirements.txt

**Monitoreo de deriva (*drift*).** Un modelo entrenado con datos de septiembre puede degradarse si en diciembre cambian los montos. El **PSI** (*Population Stability Index*) compara la distribución de una variable entre el entrenamiento y los datos nuevos: **< 0,10 estable · 0,10–0,25 vigilar · > 0,25 deriva significativa (reentrenar)**.

In [ ]:
df_nuevo = curso.datos_nuevo_mes()                           # mes siguiente: montos 80 % más altos (por ejemplo, temporada navideña)

### ✏️ Checkpoint 6 · ⏱ 4 min
Calcule el **PSI del monto** entre los datos de entrenamiento (`df`) y el nuevo mes (`df_nuevo`).

In [ ]:
psi_monto = curso.psi(df['___'], df_nuevo['___'])          # ✏️ esperado = entrenamiento, actual = nuevo mes
print(f'PSI del monto: {psi_monto:.3f}')
curso.checkpoint(6, psi_monto)

In [ ]:
tabla_psi = pd.Series({c: curso.psi(df[c], df_nuevo[c]) for c in cols}).round(3).sort_values(ascending=False)
tabla_psi.to_frame('PSI').assign(estado=lambda t: np.where(t['PSI'] > 0.25, '🔴 reentrenar', np.where(t['PSI'] > 0.10, '🟡 vigilar', '🟢 estable')))

### ➕ Ejemplo adicional: todo esto en integración continua (GitHub Actions)
```yaml
name: CI del modelo
on: [push]
jobs:
  pruebas:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.12"}
      - run: pip install -r requirements.txt pytest pytest-cov
      - run: pytest --cov=pagoya_utils --cov-fail-under=70     # falla si la cobertura baja de 70 %
```
Es el mismo mecanismo de su **verificador** del curso: cada guardado en GitHub dispara las pruebas.

### 💬 Pregunta 3
En su proyecto final, ¿qué prueba de datos y qué prueba de modelo agregaría? ¿Qué haría si el PSI de una variable importante supera 0,25?

**Respuesta:** ___

---
## Cierre de la clase

In [ ]:
curso.resumen(total=7)                                      # avance de checkpoints

**Para terminar:** revise los ✅ y las 3 preguntas 💬 → **Archivo → Guardar (Ctrl+S)** → en *Repositorio* **el suyo** → ruta `talleres/clase4.ipynb` → mensaje `Clase 4`.